In [9]:
import pandas as pd
import sys
import IPython
import os
import numpy as np
from pathlib import Path
from datetime import date, timedelta

ipynb_path = Path(IPython.extract_module_locals()[1]["__vsc_ipynb_file__"]).resolve()
notebook_name = "/".join(str(ipynb_path).split("/")[-5:])
notebook_dir = os.path.dirname(notebook_name)
passiv_dir = Path(notebook_dir).parent

myg_root = ipynb_path.parents[2]
python_root = ipynb_path.parents[3]
for p in (myg_root, python_root):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from _passiv.libraries import passiv_funktionen
from _passiv.libraries import passiv_import_data
from _passiv.libraries import passiv_kupons
from _passiv.libraries import passiv_rlz
from _passiv.libraries import passiv_bond_values

bm_dir = Path(passiv_dir, 'bm_files')

In [2]:
namen = ['PID1','R299','PID3','R55']
vips = ['084','3615','3249','5125']
krates=[[3, 5, 7, 10, 15, 30, 50],[7, 10, 15, 20, 25, 30, 50],[3, 5, 7, 10, 15, 30, 50],[1, 3, 5, 7, 10, 15, 30, 50, 70]]
countries = [['AT', 'BE', 'DE', 'ES', 'FI', 'FR', 'IE', 'IT', 'NL', 'PT','SK','SI','GR'],  
             ['AT', 'BE', 'DE', 'ES', 'FI', 'FR', 'IE', 'IT', 'NL', 'PT','SK','SI','GR'],
             ['AT','BE','DE','FR','FI','NL','SK','SI'],
             ['AT','BE','DE','FR','FI','NL','SK','SI']]
mat_sect = ['None', '10+', 'None' , 'None']

In [3]:
# # Define key-rate buckets and compute settlement date (T+1 or T+3 after Friday).
# krates = [7, 10, 15, 20, 25, 30, 50]
# #krates=[3, 5, 7, 10, 15, 30, 50]
# mat_sect = '10+'
# countries = ['AT', 'BE', 'DE', 'ES', 'FI', 'FR', 'IE', 'IT', 'NL', 'PT','SK','SI','GR']

In [17]:
   
def portfolio_neues_bewertungsdatum(pf, 
                                    neues_valuation_datum, 
                                    mat_sect=None, 
                                    krates=[3, 5, 7, 10, 15, 30, 50], 
                                    aufruf_direkt= False,
                                    verbose=False):
    pf = pf.copy()


    if verbose:
        print(f"Portfolio wird mit: {neues_valuation_datum.strftime('%Y-%m-%d')} neu bewertet.")

    if 'mat_sect' not in pf.columns.to_list():
        pf['mat_sect'] = mat_sect
        if verbose:
            print(f"mat_sect column not found in pf, setting it to {mat_sect}")
    else:
        if verbose:
            print(f"mat_sect column found in Portfolio, using existing values.")

    # wie schaut das Portfolio nach Coupon-Tag aus würden die Coupons nicht gezahlt werden
    pf['weight']  = pf['weight'].astype('float64')

    # fonds_cash = 100 - pf['weight'].sum()
    # if fonds_cash>0:
    #     if verbose:
    #         print(f"Adding cash row with weight {fonds_cash:.2f} % to portfolio.")
    #         print(f"Cash mat_sect: {mat_sect if mat_sect is not None else 'Cash'}")
    #         print(f"Cash country: DE")
    #     krd_columns = [item for item in pf.columns.to_list() if 'krd' in item]

    #     cash_row = passiv_optimize.add_cash_row2(
    #         datum=neues_valuation_datum,
    #         country='DE',
    #         cash_gewicht=fonds_cash,
    #         krd_columns=krd_columns,
    #         cash_interest= 0,
    #         mat_sect="Cash" if mat_sect is None else mat_sect
    #     )

    #     felder = ['liquidity','yield_sm','mod_dur','diff','convexity']
    #     for feld in felder:
    #         if feld not in pf.columns.to_list():
    #             cash_row = cash_row.drop(columns=[feld])

    #     pf = pf.astype({col: float for col in pf.select_dtypes(include=['float32']).columns})
    #     pf = pd.concat([pf, cash_row], ignore_index=True)
        
    # else:
    #     if verbose:
    #         print(f"No cash row added. Total weight is {pf['weight'].sum():.2f}.")


    pf = pf.astype({col: float for col in pf.select_dtypes(include=['float32']).columns})
        
    pf['rlz_neu'] = pf.apply(lambda row: passiv_rlz.rlz(start_date=neues_valuation_datum, end_date=row['maturity'], method='anniversary'), axis=1)
    pf['rlz_neu'] = np.where(pf['rlz_neu']<0, 0.000001, pf['rlz_neu']) #wenn die Restlaufzeit negativ wird, dann auf 0 setzen

    pf['first_coupon_type'] = (pf.apply(lambda row: passiv_bond_values.deduce_coupon_type(settlement_date=neues_valuation_datum,
                                                                                              maturity_date=row['maturity'], 
                                                                                              coupon=row['coupon'], yld=row['yield'], 
                                                                                              freq=row['freq'], 
                                                                                              target_mac_dur=row['mac_dur'], 
                                                                                              target_price=row['price'], 
                                                                                              issue_date=row['issue_date'], 
                                                                                              compounding='annual',
                                                                                              discount_method='standard'), axis=1))

    pf[['mac_dur_new', 'price_new']] = (pf.apply(lambda row: passiv_bond_values.calculate_macaulay_duration(settlement_date=neues_valuation_datum,
                                                                                       maturity_date=row['maturity'],
                                                                                       coupon=row['coupon']/100, 
                                                                                       yld=row['yield']/100,
                                                                                       freq=row['freq'],
                                                                                       first_coupon_type=row['first_coupon_type'],
                                                                                       issue_date=row['issue_date'],
                                                                                       compounding='annual',
                                                                                       discount_method='standard'), axis=1,result_type='expand'))

    pf['acc_int_new'] = pf.apply(lambda row: passiv_bond_values.calculate_accrued_interest(settlement_date=neues_valuation_datum,
                                                                                       maturity_date=row['maturity'],
                                                                                       coupon=row['coupon']/100, 
                                                                                       freq=row['freq'],
                                                                                       first_coupon_type=row['first_coupon_type'],
                                                                                       issue_date=row['issue_date']), axis=1)

    return pf
    
    pf['acc_additional'] = pf['coupon'].div(365)*days_to_coupon #was fällt an Stückzinsen in den Tagen bis zum kupon an
    pf['accr_int_new'] = pf['accr_int'] + pf['acc_additional'] #die neuen Stückzinsen sind die alten + das was in den nächsten Tagen noch dazu kommt
    pf['dirty_theo_new'] = pf['price_new'] + pf['accr_int_new'] #die neuen Dirty Prices sind die alten Preise + die neuen Stückzinsen
    pf['faktor'] = pf['dirty_theo_new'].div(pf['price_new']+pf['accr_int']) #der Faktor ist das Verhältnis der neuen Dirty Prices zu den alten Dirty Prices
    pf['weight_faktor'] = pf['weight'] * pf['faktor'] #wieviel verändert sich das Gewicht durch die neuen Dirty Prices
    pf['weight_new'] = pf['weight_faktor'] / pf['weight_faktor'].sum() * 100 #auf 100 normiertes Gewicht nach Coupon-Tag

    #pf['mac_dur'] = pf['mac_dur_neu'].copy()
    pf['rlz'] = pf['rlz_neu'].copy()
    pf['weight'] = pf['weight_new'].copy()
    pf['accr_int'] = pf['accr_int_new'].copy()
    pf['dirty_theo'] = pf['dirty_theo_new'].copy()

    pf = pf.drop(columns=['acc_additional',
                          'accr_int_new',
                          'dirty_theo_new',
                          'faktor',
                          'weight_faktor',
                          'weight_new',
                          'rlz_neu',
                          'mac_dur_neu',
                          'payment']) 

    if aufruf_direkt:
        if (mat_sect is not None) and (mat_sect != 'None'):
                pf = pf.query("mat_sect in @mat_sect").reset_index(drop=True)
                pf['weight'] = pf['weight']/pf['weight'].sum()*100
    
        krd_columns = [item for item in pf.columns.to_list() if 'krd' in item]
        pf = pf.drop(columns=krd_columns)
        krds = ['krd' + (f'0{item}' if item < 10 else str(item)) for item in krates]
        pf[krds] = pf.apply(lambda row: passiv_import_data.keydur3(ttm=row['rlz'],
                        coupon=row['coupon'],
                        yld=row['yield'],
                        frq=row['freq'],
                        dur_target=row['mac_dur'],
                        krates=krates), axis=1, result_type='expand')
    return pf


In [ ]:

def portfolio_neu(pf, datum, mat_sect=None, krates=[3, 5, 7, 10, 15, 30, 50], verbose=False):
    pf = pf.copy()
    pd_datum = pd.to_datetime(datum, format='%Y%m%d')
    days_to_coupon = 1
    if pd_datum.weekday() == 4:  # Friday
        days_to_coupon = 3
    coupon_trade_day = pd_datum + pd.Timedelta(days=days_to_coupon)
    pf = pf.copy()
    pf = portfolio_verschieben(pf, datum=datum, mat_sect=mat_sect, krates=krates, verbose=verbose)
    # jetzt wird das Ländergewicht fixiert und die zahlenden Kupons auf 0 gesetzt. 
    # die resultierenden Gewichte werden auf das ursprüngliche Ländergewicht rebasiert.

    pf['mac_dur_neu'] = pf.apply(lambda row: passiv_funktionen.calculate_bond_values(coupon=row['coupon'],ytm= row['yield'],rlz= row['rlz'],tilgung= 100, freq=row['freq'])['mac_dur'], axis=1)
    pf['price_theo'] = pf.apply(lambda row: passiv_funktionen.calculate_bond_values(coupon=row['coupon'],ytm= row['yield'],rlz= row['rlz'],tilgung= 100, freq=row['freq'])['price_theo'], axis=1)
    pf['country_target'] = pf.groupby("country")["weight"].transform("sum")

    pf_coupon_dates = passiv_funktionen.collect_coupon_payment_days(pf)
    cpn_list = pf_coupon_dates.loc[pf_coupon_dates["coupon_trade_day"].eq(pd.Timestamp(coupon_trade_day)), "isin"].tolist()

    pf['payment'] = np.where(pf['isin'].isin(cpn_list), 1, 0)
    pf['accr_int_new'] =np.where(pf['payment']==1, 0, pf['accr_int'])

    pf['dirty_theo_new'] = pf['price_theo'] + pf['accr_int_new']
    pf['faktor'] = pf['dirty_theo_new'].div(pf['price_theo']+pf['accr_int'])
    pf['weight_faktor'] = pf['weight'] * pf['faktor']

    # weight_new: Anteil von weight_faktor pro Land × Zielgewicht des Landes
    pf['weight_new'] = (pf['weight_faktor'] / pf.groupby('country')['weight_faktor'].transform('sum')) * pf['country_target']
    pf['weight'] = pf['weight_new'].copy()

    pf['dirty_theo'] = pf['dirty_theo_new'].copy()
    pf['accr_int'] = pf['accr_int_new'].copy()
    pf['dirty_theo'] = pf['dirty_theo_new'].copy()
    pf['mac_dur'] = pf['mac_dur_neu'].copy()
    

    pf = pf.drop(columns=['dirty_theo_new','accr_int_new','faktor','weight_faktor','weight_new','country_target','payment','mac_dur_neu'])

    if (mat_sect is not None) and (mat_sect != 'None'):
        pf = pf.query("mat_sect in @mat_sect").reset_index(drop=True)
        pf['weight'] = pf['weight']/pf['weight'].sum()*100

    krd_columns = [item for item in pf.columns.to_list() if 'krd' in item]
    pf = pf.drop(columns=krd_columns)
    krds = ['krd' + (f'0{item}' if item < 10 else str(item)) for item in krates]
    pf[krds] = pf.apply(lambda row: passiv_import_data.keydur3(ttm=row['rlz'],
                 coupon=row['coupon'],
                 yld=row['yield'],
                 frq=row['freq'],
                 dur_target=row['mac_dur'],
                 krates=krates), axis=1, result_type='expand')

    return pf


In [4]:
bm_gesamt['bloomberg'] = bm_gesamt.apply(lambda row: passiv_funktionen.get_year_fraction_exact(start_date='2027-02-14', end_date=row['maturity'], method='bloomberg'), axis=1)
bm_gesamt['anniversary'] = bm_gesamt.apply(lambda row: passiv_funktionen.get_year_fraction_exact(start_date='2027-02-14', end_date=row['maturity'], method='anniversary'), axis=1)
bm_gesamt['isda'] = bm_gesamt.apply(lambda row: passiv_funktionen.get_year_fraction_exact(start_date='2027-02-14', end_date=row['maturity'], method='isda'), axis=1)

NameError: name 'bm_gesamt' is not defined

In [12]:
bm_gesamt[['isin', 'country','maturity', 'bloomberg', 'anniversary', 'isda']].query("country=='DE'").tail(20)

,isin,country,maturity,bloomberg,anniversary,isda
124,DE0001102549,DE,2036-05-15,9.248460,9.248634,9.248305
125,DE000BU2Z072,DE,2036-08-15,9.500342,9.500000,9.499671
126,DE0001135275,DE,2037-01-04,9.889117,9.887978,9.887671
127,DE0001102598,DE,2038-05-15,11.247091,11.246575,11.246575
128,DE0001135325,DE,2039-07-04,12.383299,12.383562,12.383562
129,DE0001135366,DE,2040-07-04,13.385352,13.385246,13.384917
130,DE000BU2F009,DE,2041-05-15,14.247775,14.246575,14.246575
131,DE000BU3F007,DE,2041-05-15,14.247775,14.246575,14.246575
132,DE0001135432,DE,2042-07-04,15.383984,15.383562,15.383562
133,DE0001135481,DE,2044-07-04,17.385352,17.385246,17.384917


In [5]:
#Select Coupon Date
#Wähle das coupon_datum

from datetime import date
coupon_date = '2026-08-15' #input("Bitte geben Sie das Coupon-Datum ein (Format: YYYY-MM-DD): ")
#coupon_date = date.today()
coupon_date = pd.to_datetime(coupon_date, format='%Y-%m-%d').date()
feiertage = list(passiv_funktionen.target2_holidays(coupon_date.year).keys())
coupon_date_real = coupon_date
while coupon_date_real in feiertage:
    print(f"Das Coupon-Datum {coupon_date} ist ein Target2 - Feiertag.")
    coupon_date_real = coupon_date + pd.Timedelta(days=1)
if (coupon_date_real.weekday() == 5) or (coupon_date_real.weekday() == 6):  # Samstag oder Sonntag
    print(f"Das Coupon-Datum {coupon_date_real} fällt auf ein Wochenende.")
    coupon_date_real = coupon_date_real + pd.Timedelta(days=(7 - coupon_date_real.weekday()))
print(f"Das offizielle Coupon-Datum ist: {coupon_date}, das ist der Wochentag: {coupon_date.strftime('%A')}.")
print(f"Das reale Coupon-Datum ist: {coupon_date_real}, das ist der Wochentag: {coupon_date_real.strftime('%A')}.")

Das Coupon-Datum 2026-08-15 fällt auf ein Wochenende.
Das offizielle Coupon-Datum ist: 2026-08-15, das ist der Wochentag: Saturday.
Das reale Coupon-Datum ist: 2026-08-17, das ist der Wochentag: Monday.


In [6]:
#Handels Datum berechnen
trading_date = coupon_date_real - pd.Timedelta(days=2)
if coupon_date_real.weekday() == 0:      # Montag -> Trade-Datum war Freitag
    trading_date = coupon_date_real - pd.Timedelta(days=4)
if coupon_date_real.weekday() == 1:      # Dienstag -> Trade-Datum war Freitag
    trading_date = coupon_date_real - pd.Timedelta(days=4)
if coupon_date_real.weekday() == 6:      # Sonntag -> Trade-Datum war Freitag
    trading_date = coupon_date_real - pd.Timedelta(days=3)
print(f"Das Datum an dem ein Kupon am Coupon-Datum {coupon_date_real} {coupon_date_real.strftime('%A')} zu handeln ist, ist voraussichtlich: {trading_date} {trading_date.strftime('%A')}")

Das Datum an dem ein Kupon am Coupon-Datum 2026-08-17 Monday zu handeln ist, ist voraussichtlich: 2026-08-13 Thursday


In [7]:
# passendes Benchmark Datum für das Handelsdatum berechnen
if trading_date.weekday() == 0:      # Montag -> Benchmark-Datum war Freitag
    benchmark_date = trading_date - pd.Timedelta(days=3)
if trading_date.weekday() == 6:      # Sonntag -> Benchmark-Datum war Freitag
    benchmark_date = trading_date - pd.Timedelta(days=2)
if trading_date.weekday() != 0 and trading_date.weekday() != 6:  # Dienstag bis Samstag
    benchmark_date = trading_date - pd.Timedelta(days=1)
print(f"Das naheliegende Benchmark-Datum für das Handelsdatum {trading_date} {trading_date.strftime('%A')} ist voraussichtlich: {benchmark_date} {benchmark_date.strftime('%A')}")

benchmark_date_str = benchmark_date.strftime('%Y%m%d')
bm_exists = passiv_import_data.__find_datei(datum=benchmark_date_str, verbose=False)
if bm_exists[1]==0:
    benchmark_date_found = bm_exists[0]
else:
    benchmark_date_found = pd.to_datetime(benchmark_date_str, format='%Y%m%d')
print(f"dazu ist das nächste existierende BM Datum: {pd.to_datetime(benchmark_date_found).strftime('%Y-%m-%d')} {pd.to_datetime(benchmark_date_found).strftime('%A')}")


coupon_date_str = coupon_date.strftime('%Y%m%d')
benchmark_date_found = pd.to_datetime(benchmark_date_found).strftime('%Y%m%d')

Das naheliegende Benchmark-Datum für das Handelsdatum 2026-08-13 Thursday ist voraussichtlich: 2026-08-12 Wednesday
dazu ist das nächste existierende BM Datum: 2026-08-06 Thursday


In [10]:
for x, fonds in enumerate(namen[0:1]):
    # Keep per-fund values separate so source config variables are not overwritten.
    krates_fonds = krates[x] if isinstance(krates[0], (list, tuple, np.ndarray)) else krates
    mat_sect_fonds = mat_sect[x] #if isinstance(mat_sect, (list, tuple, np.ndarray)) else mat_sect
    countries_fonds = countries[x] if isinstance(countries[0], (list, tuple, np.ndarray)) else countries
    print(x)
    print(krates_fonds, countries_fonds, mat_sect_fonds)

0
[3, 5, 7, 10, 15, 30, 50] ['AT', 'BE', 'DE', 'ES', 'FI', 'FR', 'IE', 'IT', 'NL', 'PT', 'SK', 'SI', 'GR'] None


In [12]:
bm_gesamt = passiv_import_data.import_data(datum=benchmark_date_found, verbose=False)

In [19]:
portfolio_neues_bewertungsdatum(bm_gesamt, neues_valuation_datum=pd.to_datetime('2026-08-15'), mat_sect=mat_sect_fonds, krates=krates_fonds, verbose=True)

Portfolio wird mit: 2026-08-15 neu bewertet.
mat_sect column found in Portfolio, using existing values.


,isin,bond_description,liquidity,maturity,mat_sect,price,accr_int,country,weight,rlz,...,krd07,krd10,krd15,krd30,krd50,rlz_neu,first_coupon_type,mac_dur_new,price_new,acc_int_new
0,AT0000A1VGK0,Austria 0.5000% RAGB Apr 2027,Traded,2027-04-20,1-3,98.57200,0.153425,AT,0.186922,0.693151,...,0.000000,0.000000,0.000000,0.000000,0.0,0.679452,regular,0.67945,99.98571,0.001603
1,AT0000383864,Austria 6.2500% RAGB Jul 2027,Traded,2027-07-15,1-3,103.23200,0.445205,AT,0.122997,0.928767,...,0.000000,0.000000,0.000000,0.000000,0.0,0.915068,regular,0.91507,100.03271,0.005308
2,AT0000A1ZGE4,Austria 0.7500% RAGB Feb 2028,Traded,2028-02-20,1-3,97.05000,0.351370,AT,0.168077,1.530055,...,0.000000,0.000000,0.000000,0.000000,0.0,1.516393,regular,1.51773,99.96978,0.003616
3,AT0000A2VB47,Austria 0.0000% RAGB Oct 2028,Traded,2028-10-20,1-3,94.11900,0.000000,AT,0.132716,2.194521,...,0.000000,0.000000,0.000000,0.000000,0.0,2.180822,regular,2.18082,99.93896,0.000000
4,AT0000A269M8,Austria 0.5000% RAGB Feb 2029,Traded,2029-02-20,1-3,94.38900,0.234247,AT,0.178504,2.531507,...,0.000000,0.000000,0.000000,0.000000,0.0,2.517808,regular,2.51766,99.94143,0.002411
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,SK4000026845,Slovakia 3.7500% SLOVGB Feb 2040,Active,2040-02-27,10+,96.96875,1.684932,SK,0.035468,13.549180,...,0.518424,3.355499,6.175226,0.000000,0.0,13.535519,regular,13.50292,99.96043,0.017363
487,SK4000022547,Slovakia 4.0000% SLOVGB Feb 2043,Traded,2043-02-23,10+,97.37500,1.841096,SK,0.027035,16.539726,...,0.542822,1.097312,8.952137,0.909885,0.0,16.526027,regular,16.47174,99.96345,0.018959
488,SK4000028858,Slovakia 4.1250% SLOVGB Feb 2046,Traded,2046-02-19,10+,97.12501,1.943836,SK,0.023745,19.528767,...,0.555726,1.118952,8.222336,2.888122,0.0,19.515068,regular,19.43690,99.95731,0.020003
489,SK4120013400,Slovakia 2.0000% SLOVGB Oct 2047,Benchmark,2047-10-17,10+,67.68750,1.627397,SK,0.036558,21.185792,...,0.384630,0.773717,9.045782,5.456489,0.0,21.172131,regular,21.12632,99.50090,0.016548


In [8]:
for x, fonds in enumerate(namen[0:1]):
    # Keep per-fund values separate so source config variables are not overwritten.
    krates_fonds = krates[x] if isinstance(krates[0], (list, tuple, np.ndarray)) else krates
    mat_sect_fonds = mat_sect[x] #if isinstance(mat_sect, (list, tuple, np.ndarray)) else mat_sect
    countries_fonds = countries[x] if isinstance(countries[0], (list, tuple, np.ndarray)) else countries
    print(x)
    print(krates_fonds, countries_fonds, mat_sect_fonds)


    # if mat_sect_fonds is not None and str(mat_sect_fonds).strip().lower() != "none":
    #     mat_sect_fonds = str(mat_sect_fonds).strip()
    # else:
    #     mat_sect_fonds = None

    print(f"--- {fonds} ---")
    bm_gesamt = passiv_import_data.import_data(krates=krates_fonds, mat_sect=None, countries=countries_fonds, datum=benchmark_date_found)
    if mat_sect_fonds is not None:
        bm_mat_sect = passiv_import_data.import_data(krates=krates_fonds, mat_sect=mat_sect_fonds, countries=countries_fonds, datum=benchmark_date_found)
    else:
        bm_mat_sect = bm_gesamt.copy()

    print(f"in der Benchmark Datei für das Datum {benchmark_date_found} sind die folgenden Coupon-Daten für den {coupon_date} enthalten:")

    cp_in_bm_gesamt = (passiv_funktionen.collect_coupon_payment_days(bm_gesamt, shift_past_coupon_day_to_next_year=False)
                                    .query("coupon_payment_date_for_year == @coupon_date")
                                    .sort_values("coupon_trade_day")["isin"]
                                    .to_list())
    cp_in_bm = (passiv_funktionen.collect_coupon_payment_days(bm_mat_sect, shift_past_coupon_day_to_next_year=False)
                                    .query("coupon_payment_date_for_year == @coupon_date")
                                    .sort_values("coupon_trade_day")["isin"]
                                    .to_list())

    print(f"Anzahl der Isin mit Kupons: {len(cp_in_bm_gesamt)} in der Benchmark-Datei und {len(cp_in_bm)} in der Benchmark-Datei mit Mat-Sect {mat_sect_fonds}.")
    print("Die folgenden Isin mit Kupons sind in der Gesamt Benchmark enthalten:")
    print(bm_gesamt.query("isin in @cp_in_bm_gesamt").sort_values("weight", ascending=False)[["isin", "bond_description", "weight", "rlz"]].reset_index(drop=True))
    print()
    print("Die folgenden Isin mit Kupons sind in der Benchmark-Datei mit Mat-Sect enthalten:")
    print(bm_mat_sect.query("isin in @cp_in_bm").sort_values("weight", ascending=False)[["isin", "bond_description", "weight", "rlz"]].reset_index(drop=True))


0
[3, 5, 7, 10, 15, 30, 50] ['AT', 'BE', 'DE', 'ES', 'FI', 'FR', 'IE', 'IT', 'NL', 'PT', 'SK', 'SI', 'GR'] None
--- PID1 ---
in der Benchmark Datei für das Datum 20260806 sind die folgenden Coupon-Daten für den 2026-08-15 enthalten:
Anzahl der Isin mit Kupons: 15 in der Benchmark-Datei und 15 in der Benchmark-Datei mit Mat-Sect None.
Die folgenden Isin mit Kupons sind in der Gesamt Benchmark enthalten:
            isin                bond_description    weight        rlz
0   DE000BU2Z031  Germany 2.6000%  DBR  Aug 2034  0.417959   8.013699
1   DE000BU2Z056  Germany 2.6000%  DBR  Aug 2035  0.398381   9.013661
2   DE0001102341  Germany 2.5000%  DBR  Aug 2046  0.391287  20.013699
3   DE000BU2Z015  Germany 2.6000%  DBR  Aug 2033  0.366969   7.013699
4   DE0001102432  Germany 1.2500%  DBR  Aug 2048  0.335929  22.013699
5   DE0001102424  Germany 0.5000%  DBR  Aug 2027  0.330126   1.013661
6   DE000BU2D012  Germany 2.9000%  DBR  Aug 2056  0.321451  30.013699
7   DE0001102606  Germany 1.7000% 

In [22]:
bm_verschoben = portfolio_verschieben(bm_gesamt, datum=trading_date, mat_sect=mat_sect_fonds, aufruf_direkt=True, verbose=True)


Settlement Date: 2026-08-18 (T+4)
mat_sect column found in Portfolio, using existing values.
No cash row added. Total weight is 100.00.
Coupon list: []


In [23]:

print(passiv_funktionen.krd_table_diff(bm=bm_verschoben, fonds=bm_neu, krates=krates_fonds, runden=3))
print()

NameError: name 'bm_neu' is not defined

In [71]:
x=bm_neu[['isin','country','mac_dur','bond_description','weight']].merge(bm_verschoben[['isin','mac_dur','weight']], on='isin', suffixes=('_neu', '_verschoben')).query("country=='DE'").tail(20)

In [72]:
x

,isin,country,mac_dur_neu,bond_description,weight_neu,mac_dur_verschoben,weight_verschoben
125,DE000BU2Z072,DE,8.543895,Germany 3.0000% DBR Aug 2036,1.426641e-01,8.543895,1.426641e-01
126,DE0001135275,DE,8.605093,Germany 4.0000% DBR Jan 2037,3.258263e-01,8.605093,3.258263e-01
127,DE0001102598,DE,11.028911,Germany 1.0000% DBR May 2038,3.026184e-01,11.028911,3.026184e-01
128,DE0001135325,DE,10.302853,Germany 4.2500% DBR Jul 2039,1.948531e-01,10.302853,1.948531e-01
129,DE0001135366,DE,10.714920,Germany 4.7500% DBR Jul 2040,2.565492e-01,10.714920,2.565492e-01
130,DE000BU2F009,DE,12.232950,Germany 2.6000% DBR May 2041,2.731265e-01,12.232950,2.731265e-01
131,DE000BU3F007,DE,12.235337,Germany 2.600% DBR Apr 2041,5.197431e-02,12.235337,5.197431e-02
132,DE0001135432,DE,12.583150,Germany 3.2500% DBR Jul 2042,2.168393e-01,12.583150,2.168393e-01
133,DE0001135481,DE,14.326284,Germany 2.5000% DBR Jul 2044,3.593415e-01,14.326284,3.593415e-01
134,DE0001102341,DE,15.151800,Germany 2.5000% DBR Aug 2046,3.906892e-01,15.151800,3.906892e-01


In [2]:
tests = [
    (27.5000, "exact coupon date (semiannual boundary)"),
    (27.5001, "just after coupon"),
    (27.4999, "just before coupon"),
    (27.0000, "exact coupon date"),
    (27.0001, "just after coupon"),
    (26.9999, "just before coupon"),
]

print(f"{'rlz':>10} | {'label':<32} | {'accr_int':>10} | {'mac_dur':>10} | {'mod_dur':>10}")
print('-' * 86)
for rlz, label in tests:
    r = passiv_funktionen.calculate_bond_values(rlz=rlz, coupon=1.8, ytm=3.60903, freq=2)
    print(f"{rlz:10.4f} | {label:<32} | {r['accr_int']:10.6f} | {r['mac_dur']:10.6f} | {r['mod_dur']:10.6f}")

       rlz | label                            |   accr_int |    mac_dur |    mod_dur
--------------------------------------------------------------------------------------
   27.5000 | exact coupon date (semiannual boundary) |   0.000000 |  20.375594 |  20.014431
   27.5001 | just after coupon                |   0.000000 |  20.613471 |  20.248091
   27.4999 | just before coupon               |   0.000000 |  20.375594 |  20.014431
   27.0000 | exact coupon date                |   0.000000 |  20.133367 |  19.776497
   27.0001 | just after coupon                |   0.000000 |  20.375594 |  20.014431
   26.9999 | just before coupon               |   0.000000 |  20.133367 |  19.776497


In [4]:
tests_jump = [26.9990, 26.9995, 27.0000, 27.0005, 27.0010]
print(f"{'rlz':>8} | {'accr_int':>10} | {'mac_dur':>10}")
print('-' * 36)
for rlz in tests_jump:
    r = passiv_funktionen.calculate_bond_values(rlz=rlz, coupon=1.8, ytm=3.60903, freq=2)
    print(f"{rlz:8.4f} | {r['accr_int']:10.6f} | {r['mac_dur']:10.6f}")

     rlz |   accr_int |    mac_dur
------------------------------------
 26.9990 |   0.001800 |  20.132367
 26.9995 |   0.000900 |  20.132867
 27.0000 |   0.000000 |  20.133367
 27.0005 |   0.899100 |  19.876094
 27.0010 |   0.898200 |  19.876594


In [2]:
classic = passiv_funktionen.calculate_bond_values(rlz=27.0001, coupon=1.8, ytm=3.60903, freq=2)
ql_based = passiv_funktionen.calculate_bond_values_quantlib(rlz=27.0001, coupon=1.8, ytm=3.60903, freq=2)

print('classic:', classic)
print('quantlib:', ql_based)

classic: {'price_theo': 69.059607, 'dirty_theo': 69.059607, 'accr_int': 0.0, 'mac_dur': 20.375594, 'mod_dur': 20.014431}
quantlib: {'price_theo': 69.059607, 'dirty_theo': 69.059607, 'accr_int': 0.0, 'mac_dur': 20.375594, 'mod_dur': 20.014431}
